# Processing Sentinel-1 TOPS InSAR stack with ISCE3 & COMPASS
<br>  

**Author:** Zhenli Tang, Zhang Yunjun, August 3-7, 2026 [EarthScope InSAR Short Course (ISCE+)](https://www.earthscope.org/event/2026-technical-course-insar-processing-and-analysis-isce/).

---

## 0. Initial setup

Set up the Python environment, project paths, SARForge toolchain, and define
the area of interest (AOI). All common imports are consolidated here to avoid
redundancy in downstream cells.


In [ ]:
# === Standard library ===
import gc, os, re, glob, stat, time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlretrieve

# === Third-party ===
import numpy as np
import pandas as pd
import yaml
from compass.utils.iono import download_ionex
from matplotlib import pyplot as plt
from osgeo import gdal
plt.rcParams.update({'font.size': 12})

# === Local ===
import utils as ut

# ---------------------------------------------------------------------------
# Configuration -- dataset time/space info
# ---------------------------------------------------------------------------
start_date = '2024-07-01'
end_date   = '2024-10-11'
wsen  = (-155.50, 19.30, -154.95, 19.55)

# ---------------------------------------------------------------------------
# Configuration -- data structure
# ---------------------------------------------------------------------------
work_dir = Path('~/data/Hawaii_S1_A124').expanduser()
work_dir.mkdir(parents=True, exist_ok=True)
os.chdir(work_dir)
print('Go to directory:', work_dir)

burst_db_path = work_dir.parent / 's1-burst-db' / 'opera-burst-bbox-only.sqlite3'
slc_dir = work_dir / 'SLC'
orbit_dir = work_dir / 'orbits'
dem_path = work_dir / 'DEM' / 'cop_dem.tif'
tec_dir = work_dir / 'TEC'                # global ionospheric maps files
cslc_dir = work_dir / 'CSLC'              # coregistered SLC files
config_dir = work_dir / 'configs'         # config & log files
ifgram_dir = work_dir / 'interferograms'  # inteferograms

for d in [slc_dir, orbit_dir, tec_dir, cslc_dir, config_dir, ifgram_dir]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Start-up
# ---------------------------------------------------------------------------
MAX_CONCURRENT = 3
gc.collect()

## 1. Download SAR and auxliary data

This section prepares the data environment: burst database, SLC downloads,
orbit files, and DEM. These are prerequisites for the ISCE3 CSLC generation
and the subsequent SARForge InSAR pipeline.


### 1.0 Prepare OPERA Sentinel-1 burst ID database

Download and prepare the OPERA burst database required for Sentinel-1 burst
identification. The database (`opera-burst-bbox-only.sqlite3`) maps burst
IDs to geographic bounding boxes, enabling spatial queries for CSLC generation.


In [ ]:
# Download the pre-built OPERA burst ID database from GitHub
if not burst_db_path.exists():
    print('Downloading pre-built OPERA burst ID database from GitHub...')
    os.makedirs(os.path.dirname(burst_db_path), exist_ok=True)
    BURST_DB_URL = 'https://github.com/opera-adt/burst_db/releases/download/v0.10.0/opera-burst-bbox-only.sqlite3'
    urlretrieve(BURST_DB_URL, burst_db_path)
    print('OPERA burst ID database downloaded successfully!')
else:
    print(f'OPERA burst ID database already exist at {burst_db_path}.')

### 1.1 SLC Data Download

Download Sentinel-1 SLC products for the target area and time range using
`burst2stack`. This queries ASF/ESA for available scenes matching the AOI,
relative orbit, and swath specification.


In [ ]:
ext_str = " ".join([str(x) for x in wsen])

!burst2stack --rel-orbit 124 --all-anns --pols VV --swaths IW2 --output-dir {slc_dir} --start-date {start_date} --end-date {end_date} --extent {ext_str}


### 1.2 Orbit Files

Download precise orbit files (EOF) for each SLC scene using `eof`. Accurate
orbits are essential for geocoding and interferogram formation.


In [ ]:
!eof --search-path {slc_dir} --save-dir {orbit_dir} --force-asf

### 1.3 DEM Preparation

Download a digital elevation model (DEM) covering the AOI using `sardem`.
The DEM provides topographic phase removal during CSLC generation. A water-
body mask can be optionally downloaded for quality control.


In [ ]:
# use a bounding box much larger than the specified AOI above
# to cover all downloaded burst SLCs for the intermediate processing
buf = 2  # degree
dem_wsen = (np.floor(wsen[0] - buf), np.floor(wsen[1] - buf), np.ceil(wsen[2] + buf), np.ceil(wsen[3] + buf))
dem_path.parent.mkdir(parents=True, exist_ok=True)

# download Copernicus DEM
dem_wsen_str = " ".join([str(x) for x in dem_wsen])
!sardem --bbox {dem_wsen_str} --output-type float32 --output-format GTiff --data-source COP -o {dem_path}

# download NASADEM water-body mask
ut.download_nasadem_water_mask(dem_wsen, dem_path.parent)


### 1.4 IONEX TEC Download

Download IONEX Total Electron Content (TEC) files for ionospheric
phase correction using COMPASS's `download_ionex()`.  Each SLC date
gets its own daily TEC map (JPL final solution, 2-hour intervals).

Requires [NASA Earthdata Login](https://urs.earthdata.nasa.gov)
configured in `~/.netrc`:

```
machine urs.earthdata.nasa.gov login <user> password <pass>
```


In [ ]:
# discover all acquisition dates from downloaded SAFE files
date_list = ut.get_date_list(slc_dir)
print(f'Acquisition dates found ({len(date_list)}): {date_list}')

# download TEC file for each date
for i, date_str in enumerate(date_list):
    tec_file = download_ionex(date_str, str(tec_dir), sol_code='jpl')
    print(f'  {date_str}: {Path(tec_file).name}')

print('IONEX download complete.')


## 2. CSLC Generation


### 2.1 Generate config/run files for stack coregistration via `s1_geocode_stack`


In [ ]:
ext_str = " ".join([str(x) for x in wsen])
!s1_geocode_stack.py -s {slc_dir} -d {dem_path} -o {orbit_dir} -w {cslc_dir} -dx 10 -dy 20 --common-bursts-only --burst-db-file {burst_db_path} --unzipped --bbox {ext_str}


In [ ]:
!echo "========== run_files (first .sh) =========="; \
cat "$(ls {cslc_dir}/run_files/*.sh | head -1)"; 
!echo "========== runconfigs (first .yaml) =========="; cat "$(ls {cslc_dir}/runconfigs/*.yaml | head -1)"

### 2.2 Add ionospheric correction during the stack coregistration

We post-process the generated runconfig YAML files to add
the `tec_file` entry so that `s1_geocode_slc.py` applies ionospheric
correction during geocoding.


In [ ]:
# ---- Inject TEC file paths into generated runconfigs ----
runconfig_dir = cslc_dir / 'runconfigs'

# Build date -> tec_file mapping from downloaded IONEX files
tec_map = {}
for f in sorted(tec_dir.glob('*GIM.INX')):
    # Long IGS product name: IGS0OPSRAP_YYYYDDD0000_01D_02H_GIM.INX
    m = re.search(r'_(\d{4})(\d{3})\d{4}_', f.name)
    if m:
        y, doy = int(m.group(1)), int(m.group(2))
        dt = datetime(y, 1, 1) + timedelta(days=doy - 1)
        tec_map[dt.strftime('%Y%m%d')] = str(f)
        continue

for f in sorted(tec_dir.glob('jplg*.*i')):
    # Legacy name: jplgDDD0.YYi
    m = re.search(r'jplg(\d{3})0\.(\d{2})i', f.name)
    if m:
        doy, yy = int(m.group(1)), int(m.group(2))
        y = 2000 + yy
        dt = datetime(y, 1, 1) + timedelta(days=doy - 1)
        tec_map[dt.strftime('%Y%m%d')] = str(f)
print(f'IONEX date map: {len(tec_map)} files')

updated = 0
for cfg_path in sorted(runconfig_dir.glob('geo_runconfig_????????_*.yaml')):
    # Expect name format: geo_runconfig_YYYYMMDD_tRRR_BBBBBB_iwN.yaml
    parts = cfg_path.stem.split('_')
    if len(parts) < 4:
        continue
    date_str = parts[2]  # YYYYMMDD
    if date_str not in tec_map:
        continue
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    groups = cfg['runconfig']['groups']
    if 'tec_file' not in groups['dynamic_ancillary_file_group'] or groups['dynamic_ancillary_file_group']['tec_file'] == None:
        groups['dynamic_ancillary_file_group']['tec_file'] = tec_map[date_str]
        with open(cfg_path, 'w') as f:
            yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
        updated += 1
total = len(list(runconfig_dir.glob('geo_runconfig_*.yaml')))
print(f'Runconfigs updated with tec_file: {updated} / {total}')


### 2.3 Run stack coregistration

Execute the per-burst shell scripts generated by `s1_geocode_stack.py` with
bounded concurrency (`MAX_CONCURRENT = 4`). Completed outputs are detected and
skipped on re-run, supporting resumption after interruption.

Output: `./CSLC/{burst_id}/{date}/{burst_id}_{date}.h5`


In [ ]:
# grab all run_files
run_dir = cslc_dir / 'run_files'
run_files = [str(x) for x in run_dir.glob('run_*.sh')]
# change file permission to executables
for run_file in run_files:
    current_permissions = os.stat(run_file).st_mode
    new_permissions = current_permissions | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH
    os.chmod(run_file, new_permissions)

# run
for i, run_file in enumerate(run_files):
    print('-'*20 + f'generating the {i+1}/{len(run_files)} GSLC burst...')

    # grab output CSLC file path
    prefix = os.path.splitext(os.path.basename(run_file))[0]
    date_str = prefix.split('_')[1]
    burst_id = prefix.split(date_str)[1][1:]
    cslc_path = cslc_dir / burst_id / date_str / f'{burst_id}_{date_str}.h5'

    # skip re-generating if exists
    if os.path.exists(cslc_path):
        print(f'CSLC file exists at: {cslc_path}, skip re-generating.')
    else:
        !{run_file}


### 2.4 Download static layers

In [ ]:
burst_id_list = sorted(os.path.basename(d) for d in cslc_dir.iterdir() if d.is_dir() and d.name.startswith('t'))
date_list = ut.get_date_list(slc_dir)
downloaded = ut.download_opera_static_layers(
    burst_id_list, work_dir, date_list[0], bbox_wsen=wsen,
)

### 2.5 Compute baselines time series

In [ ]:
print(f'Burst IDs: {burst_id_list}')

ut.compute_baselines_for_bursts(
    burst_ids=burst_id_list,
    cslc_dir = cslc_dir,
    output_base=str(work_dir / 'baselines')
)

# average multi burst baselines into single baseline
#ut.merge_baselines(
#    baseline_dir=str(work_dir / 'baselines'),
#    output_dir=str(work_dir / 'merged'),
#)

## 3. Interferogram stack generation

The pipeline processes CSLC outputs through an 8-step workflow that
transforms raw geocoded SLCs into unwrapped interferograms ready for time-series
analysis (e.g., MintPy).

> For multi-burst data, a **three-phase architecture** is used:
> 1. **Per-burst** (3.1–3.3): Each burst processed independently
> 2. **Stitch** (3.4): Merge bursts and crop to exact AOI
> 3. **Uniform** (3.5–3.8): Post-processing on merged data


### 3.1 Crop SLC

Crop CSLC HDF5 files to the AOI bounding box plus a buffer margin. The buffer
ensures coherence estimation windows fully cover the target area without edge
effects.

- **Input**: `./CSLC/{burst_id}/{date}.h5`
- **Output**: `./cropped_slc/{burst_id}/{date}.slc.tif`
- **Key options**: `--wsen`, `--buffer 0.05`, `--max-workers`


In [ ]:
h5_files = sorted(glob.glob(str(cslc_dir / 't*iw*/*/*.h5')))
h5_files = [f for f in h5_files if 'static_layers' not in f]

tasks = []
for f in h5_files:
    parts = Path(f).parts
    burst_id = parts[-3]
    date_str = parts[-2]
    out_file = str(process_dir / 'cropped_slc' / burst_id / f'{date_str}.slc.tif')
    tasks.append((f, out_file))

def _crop_task(args):
    f, out = args
    return ut.crop_slc_single(f, out, wse, buffer=0.05)

print(f'Processing {len(tasks)} files...')
with ThreadPoolExecutor(max_workers=MAX_CONCURRENT) as ex:
    list(ex.map(_crop_task, tasks))
print('Crop complete.')

In [ ]:
burst_dir = sorted((process_dir / 'cropped_slc').iterdir())[0]
burst_id = burst_dir.name

slc_files = sorted(burst_dir.glob('*.slc.tif'))
slc_path = str(slc_files[0])

ds = gdal.Open(slc_path)
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None

amplitude = np.abs(arr)
amp_db = 10 * np.log10(np.maximum(amplitude, 1e-6))

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(amp_db, aspect='auto', cmap='gray',
               vmin=15, vmax=20)
ax.set_title('SLC Amplitude (dB)', fontsize=12)
ax.set_xlabel('Range (pixels)')
ax.set_ylabel('Azimuth (pixels)')
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('dB')
plt.tight_layout()
plt.show()


### 3.2 Select interferometric pair list

Scan cropped SLC files across all bursts and generate a list of interferometric
pairs. All bursts share the same pair list since acquisition dates are aligned.

- **Input**: `./cropped_slc/` (all burst subdirectories)
- **Output**: `./ifgrams/ifgram_list.txt`
- **Key option**: `-n N` — maximum temporal separation between pairs

In [ ]:
ut.generate_ifgram_pairs(str(process_dir / "cropped_slc"), str(process_dir / "ifgrams"), n_connections=3)

### 3.3 Generate Interferograms

Form complex interferograms from SLC pairs using `dolphin`. Each burst is
processed independently.

- **Input**: `./cropped_slc/` + pair list
- **Output**: `./ifgrams/{burst_id}/{date1}_{date2}.int.tif`
- **Key options**: `--processor isce3`, `--max-workers`


In [ ]:
pairs_df = pd.read_csv(str(process_dir / 'ifgrams/ifgram_list.txt'), comment='#', sep=r'\s+', names=['date12'])
burst_pattern = re.compile(r'^t\d+_\d+_iw\d+$')
burst_dirs = sorted(d for d in Path(str(process_dir / "cropped_slc")).iterdir()
                    if d.is_dir() and burst_pattern.match(d.name))

tasks = []
for burst_dir in burst_dirs:
    bust_out = Path(str(process_dir / "ifgrams")) / burst_dir.name
    bust_out.mkdir(parents=True, exist_ok=True)
    for _, row in pairs_df.iterrows():
        d1, d2 = row['date12'].split('-')
        ref_slc = burst_dir / f'{d1}.slc.tif'
        sec_slc = burst_dir / f'{d2}.slc.tif'
        out_path = bust_out / f'{d1}_{d2}.int.tif'
        tasks.append((str(ref_slc), str(sec_slc), str(out_path)))

def _ifg_task(args):
    r, s, o = args
    return ut.form_single_ifgram(r, s, o)

print(f'Forming {len(tasks)} interferograms...')
with ThreadPoolExecutor(max_workers=MAX_CONCURRENT) as ex:
    list(ex.map(_ifg_task, tasks))
print('Interferogram formation complete.')

In [ ]:
ref_date = slc_files[0].name.split('.')[0] 
ifg_files = sorted((process_dir / 'ifgrams' / burst_id).glob(f'{ref_date}_*.int.tif'))
ifg_path = str(ifg_files[0])

ds = gdal.Open(ifg_path)
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None

phase = np.angle(arr)
amplitude = np.abs(arr)
amp_db = 10 * np.log10(np.maximum(amplitude, 1e-6))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
im1 = ax1.imshow(phase, aspect='auto', cmap='RdBu_r',
                 vmin=-np.pi, vmax=np.pi)
ax1.set_title('Wrapped Phase', fontsize=12)
ax1.set_xlabel('Range (pixels)')
ax1.set_ylabel('Azimuth (pixels)')
cbar1 = plt.colorbar(im1, ax=ax1, shrink=0.85)
cbar1.set_label('Phase (rad)')

vmin = np.nanpercentile(amp_db, 2)
vmax = np.nanpercentile(amp_db, 98)
im2 = ax2.imshow(amp_db, aspect='auto', cmap='gray',
                 vmin=vmin, vmax=vmax)
ax2.set_title('Amplitude (dB)', fontsize=12)
ax2.set_xlabel('Range (pixels)')
cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.85)
cbar2.set_label('dB')

plt.tight_layout()
plt.show()


### 3.4 Stitch burst interferograms

Merge multi-burst interferograms into a single continuous image. The optional
`--out-bounds` crop removes the buffer region, restoring the exact AOI.

- **Input**: `./ifgrams/{burst_id}/`
- **Output**: `./stitched/ifgrams/{date1}_{date2}.int.tif`
- **Key option**: `--out-bounds` for precise bbox cropping


In [ ]:
out_bounds = wse

ut.stitch_ifgrams(str(process_dir / 'ifgrams') + '/', out_bounds, str(process_dir / 'stitched/ifgrams'))

In [ ]:
ifg_files = sorted((process_dir / 'stitched' / 'ifgrams').glob(f'{ref_date}_*.int.tif'))
ifg_path = str(ifg_files[0])

ds = gdal.Open(ifg_path)
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None

phase = np.angle(arr)
data = phase

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(data, aspect='auto', cmap='RdBu_r',
               vmin=-np.pi, vmax=np.pi)
ax.set_title('Wrapped Phase (stitched)', fontsize=12)
ax.set_xlabel('Range (pixels)')
ax.set_ylabel('Azimuth (pixels)')
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Phase (rad)')
plt.tight_layout()
plt.show()


### 3.5 Multilooking

Apply spatial averaging (multilooking) to reduce resolution and suppress phase
noise. From this step onward, all processing operates on the unified stitched
data.

- **Input**: `./stitched/ifgrams/*.int.tif`
- **Output**: `./stitched/ifgrams_multilooked/multilooked_{d1}_{d2}.int.tif`
- **Key options**: `--lks-y`, `--lks-x`, `--method`


In [ ]:
files = sorted(glob.glob(str(process_dir / 'stitched/ifgrams/*.int.tif')))
Path(str(process_dir / 'stitched/ifgrams_multilooked')).mkdir(parents=True, exist_ok=True)

def _ml_task(f):
    out = str(process_dir / 'stitched/ifgrams_multilooked') + "/" + f"multilooked_{Path(f).name}"
    return ut.multilook_tif(f, out, lks_y=2, lks_x=4, method='mean')

print(f'Multilooking {len(files)} files...')
with ThreadPoolExecutor(max_workers=MAX_CONCURRENT) as ex:
    list(ex.map(_ml_task, files))
print('Multilook complete.')

In [ ]:
ifg_files = sorted((process_dir / 'stitched' / 'ifgrams_multilooked').glob(f'*{ref_date}_*.int.tif'))
ifg_path = str(ifg_files[0])

ds = gdal.Open(ifg_path)
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None

phase = np.angle(arr)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(phase, aspect='auto', cmap='RdBu_r',
               vmin=-np.pi, vmax=np.pi)
ax.set_title('Wrapped Phase (multilooked)', fontsize=12)
ax.set_xlabel('Range (pixels)')
ax.set_ylabel('Azimuth (pixels)')
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Phase (rad)')
plt.tight_layout()
plt.show()


### 3.6 Filtering

Apply adaptive Goldstein phase filtering to reduce interferometric noise while
preserving fringe structure. The filter adapts its strength based on local
coherence.

- **Input**: `./stitched/ifgrams_multilooked/*.int.tif`
- **Output**: `./stitched/ifgrams_filtered/filtered_multilooked_{d1}_{d2}.int.tif`
- **Key option**: `--alpha` (default 0.5)


In [ ]:
files = sorted(glob.glob(str(process_dir / 'stitched/ifgrams_multilooked/*.int.tif')))
Path(str(process_dir / 'stitched/ifgrams_filtered')).mkdir(parents=True, exist_ok=True)

def _filt_task(f):
    out = str(process_dir / 'stitched'/'ifgrams_filtered') + "/" + f"filtered_{Path(f).name}"
    return ut.filter_tif(f, out, alpha=0.5)

print(f'Filtering {len(files)} files...')
with ThreadPoolExecutor(max_workers=MAX_CONCURRENT) as ex:
    list(ex.map(_filt_task, files))
print('Filtering complete.')

In [ ]:
ifg_files = sorted((process_dir / 'stitched' / 'ifgrams_filtered').glob(f'*{ref_date}_*.int.tif'))
ifg_path = str(ifg_files[0])

ds = gdal.Open(ifg_path)
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None

phase = np.angle(arr)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(phase, aspect='auto', cmap='RdBu_r',
               vmin=-np.pi, vmax=np.pi)
ax.set_title('Wrapped Phase (filtered)', fontsize=12)
ax.set_xlabel('Range (pixels)')
ax.set_ylabel('Azimuth (pixels)')
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Phase (rad)')
plt.tight_layout()
plt.show()


### 3.7 Estimate phase-sigma coherence

Compute phase-sigma correlation as a quality metric for each interferogram.
Unlike complex coherence (which requires raw SLC data), phase-sigma coherence
operates directly on the filtered interferogram.

- **Input**: `./stitched/ifgrams_filtered/*.int.tif`
- **Output**: `./stitched/ifgrams_filtered/*.phsig.coh.tif`
- **Key options**: `--ps-window-size`, `--skip-complex-coherence`


In [ ]:
files = sorted(glob.glob(str(process_dir / 'stitched/ifgrams_filtered/*.int.tif')))

def _coh_task(f):
    return ut.generate_phsig_coh_tif(f, nlks=8)

print(f'Computing coherence for {len(files)} files...')
with ThreadPoolExecutor(max_workers=MAX_CONCURRENT) as ex:
    list(ex.map(_coh_task, files))
print('Coherence computation complete.')

In [ ]:
coh_files = sorted((process_dir / 'stitched' / 'ifgrams_filtered').glob(f'*{ref_date}_*.phsig.coh.tif'))
coh_path = str(coh_files[0])

ds = gdal.Open(coh_path)
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None

data = arr

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(data, aspect='auto', cmap='gray',
               vmin=0, vmax=1)
ax.set_title('Phase-Sigma Coherence', fontsize=12)
ax.set_xlabel('Range (pixels)')
ax.set_ylabel('Azimuth (pixels)')
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Coherence')
plt.tight_layout()
plt.show()


### 3.8 Phase unwrapping

Unwrap filtered interferograms using SNAPHU, guided by the phase-sigma
coherence map. The unwrapped phase is the final product, ready for time-series
analysis with MintPy.

- **Input**: `./stitched/ifgrams_filtered/*.int.tif` + `.phsig.coh.tif`
- **Output**: `./stitched/unwrapped/{date1}_{date2}.unw.tif`


In [ ]:
ifg_files = sorted(glob.glob(str(process_dir / 'stitched/ifgrams_filtered/*.int.tif')))
unw_dir = str(process_dir / "stitched/unwrapped")
Path(unw_dir).mkdir(parents=True, exist_ok=True)

water_mask = dem_dir / 'swbd_nasadem.wbd'

def _unwrap_task(f):
    base = Path(f).name.replace('.int.tif', '')
    for prefix in ['filtered_multilooked_', 'filtered_', 'multilooked_']:
        if base.startswith(prefix):
            base = base[len(prefix):]
    coh_file = str(Path(f).parent / f'{base}.phsig.coh.tif')
    out_file = f'{unw_dir}/{base}.unw.tif'
    return ut.unwrap_single_ifgram(f, coh_file, out_file, nlooks=8,
                                cost_mode='smooth', init_method='mcf',
                                water_mask=water_mask)

print(f'Unwrapping {len(ifg_files)} files...')
with ThreadPoolExecutor(max_workers=MAX_CONCURRENT) as ex:
    list(ex.map(_unwrap_task, ifg_files))
print('Unwrapping complete.')

In [ ]:
unw_files = sorted((process_dir / 'stitched' / 'unwrapped').glob(f'*{ref_date}_*.unw.tif'))
unw_path = str(unw_files[0])

ds = gdal.Open(unw_path)
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None

data = arr

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(data, aspect='auto', cmap='jet')
ax.set_title('Unwrapped Phase', fontsize=12)
ax.set_xlabel('Range (pixels)')
ax.set_ylabel('Azimuth (pixels)')
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Phase (rad)')
plt.tight_layout()
plt.show()